# Using Captum
Captum is a model interpretability and understanding library for PyTorch. Captum means comprehension in Latin and contains general purpose implementations of integrated gradients, saliency maps, smoothgrad, vargrad and others for PyTorch models. It has quick integration for models built with domain-specific libraries such as torchvision, torchtext, and others.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import huggingface_hub
import torch

from huggingface_hub import hf_hub_download
from huggingface_hub import login
from transformers import pipeline

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from html import escape

from captum.attr import LayerIntegratedGradients

## load dataframes which we investigate

In [2]:
df_bear_but_pred_bull = pd.read_csv("../data/predictions/df_bear_but_pred_bull.csv")
df_bull_but_pred_bear = pd.read_csv("../data/predictions/df_bull_but_pred_bear.csv")

In [3]:
df_bear_but_pred_bull.head(10)

,text,label,predicted_scores,predicted_label,negative,neutral,positive,pred_top_score
0,Grubhub gains a bear on margin view,negative,"{'positive': 0.8692758679389954, 'negative': 0...",positive,0.068459,0.062265,0.869276,0.869276
1,Needham's Martin Defends Her Prediction That N...,negative,"{'positive': 0.5826396346092224, 'neutral': 0....",positive,0.092902,0.324459,0.582640,0.582640
2,Rio Tinto downgraded to neutral from overweigh...,negative,"{'positive': 0.5796186327934265, 'negative': 0...",positive,0.294971,0.125410,0.579619,0.579619
3,Royal Caribbean stock price target cut to $100...,negative,"{'positive': 0.44598066806793213, 'negative': ...",positive,0.434099,0.119920,0.445981,0.445981
4,SunTrust latest to trim Nielsen target,negative,"{'positive': 0.7235925197601318, 'neutral': 0....",positive,0.077757,0.198651,0.723593,0.723593
5,Nike to record $425 mln charge as it transitio...,negative,"{'positive': 0.7930759191513062, 'neutral': 0....",positive,0.019527,0.187397,0.793076,0.793076
6,Oracle’s Need for ‘Steep’ Sales Climb Leaves S...,negative,"{'positive': 0.6714838743209839, 'negative': 0...",positive,0.235818,0.092698,0.671484,0.671484
7,TikTok looks to diversify users as U.S. pressu...,negative,"{'positive': 0.9266785979270935, 'neutral': 0....",positive,0.008610,0.064711,0.926679,0.926679
8,Trading on the News Adds a Risk Premium to Sha...,negative,"{'positive': 0.9199829697608948, 'neutral': 0....",positive,0.025549,0.054468,0.919983,0.919983
9,Trans Mountain expansion costs soars to C$12.6...,negative,"{'positive': 0.9197039604187012, 'negative': 0...",positive,0.049697,0.030599,0.919704,0.919704


## retrieve Finbert model

In [4]:
login()

In [5]:
# get classifier
classifier = pipeline("text-classification", model="ProsusAI/finbert")
model = classifier.model
tokenizer = classifier.tokenizer

model.eval() # evaluation mode

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
# for a whole batch
def predict_batch(texts):
    preds = classifier(texts, batch_size=512, truncation=True, max_length=512, top_k=None)
    rows = []
    for row in preds:
        rows.append({d["label"]: float(d["score"]) for d in row})
    return rows

# example with dataframe texts
texts = df_bear_but_pred_bull["text"].astype(str).tolist()
predictions = predict_batch(texts)
predictions

[{'positive': 0.8692759871482849,
  'negative': 0.06845902651548386,
  'neutral': 0.06226496398448944},
 {'positive': 0.5826385021209717,
  'neutral': 0.3244595527648926,
  'negative': 0.09290196746587753},
 {'positive': 0.5796201825141907,
  'negative': 0.2949697971343994,
  'neutral': 0.12540997564792633},
 {'positive': 0.44598087668418884,
  'negative': 0.434099018573761,
  'neutral': 0.11992007493972778},
 {'positive': 0.7235931158065796,
  'neutral': 0.19865047931671143,
  'negative': 0.07775641977787018},
 {'positive': 0.7930759191513062,
  'neutral': 0.1873966008424759,
  'negative': 0.01952746883034706},
 {'positive': 0.67148357629776,
  'negative': 0.23581823706626892,
  'neutral': 0.09269817173480988},
 {'positive': 0.9266787171363831,
  'neutral': 0.06471139192581177,
  'negative': 0.008609970100224018},
 {'positive': 0.9199829697608948,
  'neutral': 0.05446789041161537,
  'negative': 0.025549182668328285},
 {'positive': 0.9197039604187012,
  'negative': 0.049696631729602814

In [7]:
# Define forward function
def forward_func(input_ids, attention_mask):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    return outputs.logits

# Access FinBERT's word embedding layer
embedding_layer = model.base_model.embeddings.word_embeddings

lig = LayerIntegratedGradients(
    forward_func,
    embedding_layer
)

def explain_tweet(tweet, n_steps=50):

    device = next(model.parameters()).device

    inputs = tokenizer(
        tweet,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Get prediction
    with torch.no_grad():
        logits = forward_func(input_ids, attention_mask)
        probabilities = torch.softmax(logits, dim=1)

    predicted_class = logits.argmax(dim=1).item()
    predicted_label = model.config.id2label[predicted_class]

    # Baseline: replace normal tokens with padding
    baseline_ids = torch.full_like(
        input_ids,
        tokenizer.pad_token_id
    )

    # Preserve special tokens
    special_tokens = tokenizer.get_special_tokens_mask(
        input_ids[0].tolist(),
        already_has_special_tokens=True
    )

    for i, is_special in enumerate(special_tokens):
        if is_special:
            baseline_ids[0, i] = input_ids[0, i]

    # Calculate attributions
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=predicted_class,
        n_steps=n_steps,
        return_convergence_delta=True
    )

    # Aggregate embedding dimensions
    scores = attributions.sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(
        input_ids[0]
    )

    print("Tweet:", tweet)
    print("Prediction:", predicted_label)
    print("Confidence:", probabilities[0, predicted_class].item())
    print("\nToken contributions:")

    for token, score in zip(tokens, scores):
        print(f"{token:20s} {score.item():+.5f}")

    print("\nConvergence delta:", delta.item())

    return {
        "tweet": tweet,
        "prediction": predicted_label,
        "confidence": probabilities[0].tolist(),
        "tokens": tokens,
        "attributions": scores.detach().cpu().tolist()
    }


# Test on a misclassified tweet
result = explain_tweet(
    df_bear_but_pred_bull["text"].iloc[9]
)

Tweet: Trans Mountain expansion costs soars to C$12.6B - report
Prediction: positive
Confidence: 0.9197039008140564

Token contributions:
[CLS]                +0.00000
trans                -0.03988
mountain             +0.02905
expansion            +0.48567
costs                +0.20672
so                   +0.11384
##ars                +0.21266
to                   +0.20806
c                    +0.06054
$                    +0.34086
12                   -0.00040
.                    +0.02922
6                    +0.33847
##b                  +0.09801
-                    +0.19800
report               -0.21585
[SEP]                +0.00000

Convergence delta: -0.13283610343933105


In [8]:
print(df_bear_but_pred_bull.iloc[9])
result

text                Trans Mountain expansion costs soars to C$12.6...
label                                                        negative
predicted_scores    {'positive': 0.9197039604187012, 'negative': 0...
predicted_label                                              positive
negative                                                     0.049697
neutral                                                      0.030599
positive                                                     0.919704
pred_top_score                                               0.919704
Name: 9, dtype: object


{'tweet': 'Trans Mountain expansion costs soars to C$12.6B - report',
 'prediction': 'positive',
 'confidence': [0.9197039008140564,
  0.049696702510118484,
  0.030599400401115417],
 'tokens': ['[CLS]',
  'trans',
  'mountain',
  'expansion',
  'costs',
  'so',
  '##ars',
  'to',
  'c',
  '$',
  '12',
  '.',
  '6',
  '##b',
  '-',
  'report',
  '[SEP]'],
 'attributions': [0.0,
  -0.039875537157058716,
  0.029054762795567513,
  0.48566919565200806,
  0.20671793818473816,
  0.11383862048387527,
  0.21266472339630127,
  0.20805734395980835,
  0.06053803116083145,
  0.3408642113208771,
  -0.0004042908549308777,
  0.0292232483625412,
  0.33846810460090637,
  0.0980079248547554,
  0.19799861311912537,
  -0.21584990620613098,
  0.0]}

In [9]:
df_bear_but_pred_bull["text"].iloc[2]

'Rio Tinto downgraded to neutral from overweight at J.P. Morgan'

## visualize tweets

In [10]:
from html import escape

def visualize_tweet(result):

    tokens = result["tokens"]
    scores = result["attributions"]

    # Normalize scores for visualization
    max_score = max(abs(s) for s in scores) or 1

    html_tokens = []

    for token, score in zip(tokens, scores):

        # Remove WordPiece notation
        word = token.replace("##", "")

        intensity = abs(score) / max_score

        if score > 0:
            color = f"rgba(0, 180, 0, {intensity:.2f})"
        else:
            color = f"rgba(220, 0, 0, {intensity:.2f})"

        html_tokens.append(
            f'<span title="Attribution: {score:.5f}" '
            f'style="background-color:{color}; '
            f'padding:4px; margin:2px; '
            f'border-radius:4px;">'
            f'{escape(word)}</span>'
        )

    highlighted_tweet = " ".join(html_tokens)

    html_content = f"""
    <html>
    <body style="font-family:Arial; padding:25px;">

        <h2>FinBERT Prediction</h2>

        <p>
            <b>Sentiment:</b> {escape(result["prediction"])}
        </p>

        <p>
            <b>Confidence:</b>
            {result["confidence"][
                list(model.config.id2label.keys()).index(
                    int(next(
                        k for k, v in model.config.id2label.items()
                        if v == result["prediction"]
                    ))
                )] * 100:.2f}%
        </p>

        <h3>Tweet explanation</h3>

        <div style="font-size:20px; line-height:2;">
            {highlighted_tweet}
        </div>

        <hr>

        <p>
            <span style="background-color:rgba(0,180,0,0.5)">
            Positive contribution
            </span>

            <span style="background-color:rgba(220,0,0,0.5)">
            Negative contribution
            </span>
        </p>

    </body>
    </html>
    """

    with open("finbert_explanation.html", "w", encoding="utf-8") as f:
        f.write(html_content)

    print("Visualization saved as finbert_explanation.html")

In [11]:
result = explain_tweet(
    df_bear_but_pred_bull["text"].iloc[9]
)

visualize_tweet(result)

Tweet: Trans Mountain expansion costs soars to C$12.6B - report
Prediction: positive
Confidence: 0.9197039008140564

Token contributions:
[CLS]                +0.00000
trans                -0.03988
mountain             +0.02905
expansion            +0.48567
costs                +0.20672
so                   +0.11384
##ars                +0.21266
to                   +0.20806
c                    +0.06054
$                    +0.34086
12                   -0.00040
.                    +0.02922
6                    +0.33847
##b                  +0.09801
-                    +0.19800
report               -0.21585
[SEP]                +0.00000

Convergence delta: -0.13283610343933105
Visualization saved as finbert_explanation.html


## interactive dashboard

In [12]:
# Get class labels in the correct numerical order
id2label = {
    int(k): v for k, v in model.config.id2label.items()
}

# --------------------------------------------------
# Dashboard components
# --------------------------------------------------

tweet_input = widgets.Textarea(
    value="The company reported excellent profits but its stock crashed.",
    placeholder="Enter a tweet or sentence to analyze...",
    description="Tweet:",
    layout=widgets.Layout(width="100%", height="110px"),
    style={"description_width": "initial"}
)

actual_label = widgets.Dropdown(
    options=[
        ("Not specified", "unknown"),
        ("Positive", "positive"),
        ("Negative", "negative"),
        ("Neutral", "neutral")
    ],
    value="unknown",
    description="Actual label:",
    style={"description_width": "initial"}
)

analyze_button = widgets.Button(
    description="Analyze Tweet",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="180px")
)

output = widgets.Output()

# --------------------------------------------------
# Dashboard visualization
# --------------------------------------------------

def create_dashboard(result, actual="unknown"):

    prediction = result["prediction"]
    confidence = result["confidence"]
    predicted_class = result.get(
        "predicted_class",
        next(
            k for k, v in id2label.items()
            if v.lower() == prediction.lower()
        )
    )

    tokens = result["tokens"]
    scores = result["attributions"]

    # Confidence scores for each class
    confidence_html = ""

    for class_id, label in id2label.items():

        probability = confidence[class_id]

        color = {
            "positive": "#16a34a",
            "negative": "#dc2626",
            "neutral": "#64748b"
        }.get(label.lower(), "#64748b")

        confidence_html += f"""
        <div style="margin:12px 0;">
            <div style="display:flex;justify-content:space-between;">
                <b>{escape(label.capitalize())}</b>
                <b>{probability * 100:.2f}%</b>
            </div>
            <div style="background:#e5e7eb;height:12px;
                        border-radius:8px;overflow:hidden;">
                <div style="background:{color};
                            width:{probability * 100:.2f}%;
                            height:12px;border-radius:8px;">
                </div>
            </div>
        </div>
        """

    # Token attribution visualization
    max_score = max((abs(s) for s in scores), default=0) or 1

    token_html = ""

    for token, score in zip(tokens, scores):

        word = token.replace("##", "")

        intensity = abs(score) / max_score

        if score > 0:
            background = (
                f"rgba(22,163,74,{0.12 + 0.75 * intensity:.2f})"
            )
        else:
            background = (
                f"rgba(220,38,38,{0.12 + 0.75 * intensity:.2f})"
            )

        token_html += f"""
        <span title="Attribution: {score:+.5f}"
              style="display:inline-block;
                     background:{background};
                     padding:5px 7px;
                     margin:3px;
                     border-radius:5px;
                     cursor:help;">
            {escape(word)}
        </span>
        """

    # Actual vs predicted comparison
    if actual != "unknown":

        correct = actual.lower() == prediction.lower()

        if correct:
            comparison = """
            <div style="background:#dcfce7;padding:12px;
                        border-radius:8px;color:#166534;">
                ✓ Prediction matches the actual sentiment.
            </div>
            """
        else:
            comparison = f"""
            <div style="background:#fee2e2;padding:12px;
                        border-radius:8px;color:#991b1b;">
                ✗ Misclassification detected!
                <br>Actual: <b>{escape(actual.capitalize())}</b>
                <br>Predicted: <b>{escape(prediction.capitalize())}</b>
            </div>
            """
    else:
        comparison = ""

    # Most influential tokens
    ranked = sorted(
        zip(tokens, scores),
        key=lambda x: abs(x[1]),
        reverse=True
    )[:8]

    ranking_html = ""

    for token, score in ranked:
        ranking_html += f"""
        <tr>
            <td style="padding:7px;">{escape(token)}</td>
            <td style="padding:7px;text-align:right;">
                {score:+.5f}
            </td>
        </tr>
        """

    # Full dashboard
    dashboard_html = f"""
    <div style="font-family:Arial,sans-serif;
                color:#1f2937;max-width:900px;">

        <h2 style="margin-top:15px;">
            FinBERT Explainability Dashboard
        </h2>

        <div style="border:1px solid #ddd;
                    border-radius:10px;padding:18px;margin:15px 0;">

            <h3>1. Classification results</h3>

            <p>
                Predicted sentiment:
                <b style="font-size:20px;">
                    {escape(prediction.upper())}
                </b>
            </p>

            <p>
                Confidence:
                <b>{confidence[predicted_class] * 100:.2f}%</b>
            </p>

            {comparison}

            <h4>Class probabilities</h4>
            {confidence_html}

        </div>

        <div style="border:1px solid #ddd;
                    border-radius:10px;padding:18px;margin:15px 0;">

            <h3>2. Word-level explanations</h3>

            <p style="line-height:2.5;font-size:17px;">
                {token_html}
            </p>

            <div style="display:flex;gap:15px;flex-wrap:wrap;">
                <span>
                    <span style="background:#16a34a;padding:5px;">
                    &nbsp;&nbsp;&nbsp;
                    </span>
                    Supports predicted class
                </span>

                <span>
                    <span style="background:#dc2626;padding:5px;">
                    &nbsp;&nbsp;&nbsp;
                    </span>
                    Opposes predicted class
                </span>
            </div>

            <p style="font-size:12px;color:#666;">
                Hover over individual words to see their attribution.
                Color intensity represents relative attribution magnitude.
            </p>

        </div>

        <div style="border:1px solid #ddd;
                    border-radius:10px;padding:18px;margin:15px 0;">

            <h3>3. Most influential tokens</h3>

            <table style="width:100%;border-collapse:collapse;">
                <tr style="background:#f3f4f6;">
                    <th style="padding:8px;text-align:left;">Token</th>
                    <th style="padding:8px;text-align:right;">
                        Attribution
                    </th>
                </tr>
                {ranking_html}
            </table>

        </div>

    </div>
    """

    display(HTML(dashboard_html))


# --------------------------------------------------
# Analyze button functionality
# --------------------------------------------------

def analyze_button_clicked(button):

    with output:
        clear_output(wait=True)

        tweet = tweet_input.value.strip()

        if not tweet:
            print("Please enter a tweet first.")
            return

        print("Analyzing tweet...")

        try:
            result = explain_tweet(tweet)

            # Add predicted class index if not already returned
            result["predicted_class"] = next(
                k for k, v in id2label.items()
                if v.lower() == result["prediction"].lower()
            )

            clear_output(wait=True)

            create_dashboard(
                result,
                actual=actual_label.value
            )

        except Exception as e:
            print("Error during analysis:", str(e))


In [13]:
analyze_button.on_click(analyze_button_clicked)

# --------------------------------------------------
# Display dashboard
# --------------------------------------------------

dashboard = widgets.VBox([
    widgets.HTML("<h2>FinBERT Tweet Analyzer</h2>"),
    tweet_input,
    actual_label,
    analyze_button,
    output
])

display(dashboard)

In [14]:
# =====================================================
# 1. DATASET CONFIGURATION
# =====================================================

datasets = {
    "Bearish → Bullish": df_bear_but_pred_bull.reset_index(drop=True),
    "Bullish → Bearish": df_bull_but_pred_bear.reset_index(drop=True)
}

# The actual sentiment and prediction columns in your data
TEXT_COL = "text"
ACTUAL_COL = "label"
PREDICTED_COL = "predicted_label"

SENTIMENT_COLS = ["negative", "neutral", "positive"]

# =====================================================
# 2. WIDGETS
# =====================================================

category_selector = widgets.Dropdown(
    options=list(datasets.keys()),
    value="Bearish → Bullish",
    description="Error type:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "initial"}
)

tweet_selector = widgets.Dropdown(
    options=[],
    description="Select tweet:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "initial"}
)

previous_button = widgets.Button(
    description="Previous",
    icon="arrow-left",
    layout=widgets.Layout(width="120px")
)

next_button = widgets.Button(
    description="Next",
    icon="arrow-right",
    layout=widgets.Layout(width="120px")
)

analyze_button = widgets.Button(
    description="Analyze tweet",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="180px")
)

dataset_stats = widgets.HTML()
tweet_counter = widgets.HTML()
analysis_output = widgets.Output()

# =====================================================
# 3. DATASET INFORMATION
# =====================================================

def get_current_dataset():
    return datasets[category_selector.value]


def update_dataset_stats():

    total_bear = len(df_bear_but_pred_bull)
    total_bull = len(df_bull_but_pred_bear)
    total = total_bear + total_bull

    dataset_stats.value = f"""
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin:15px 0;">

        <div style="flex:1;min-width:140px;background:#f1f5f9;
                    padding:15px;border-radius:10px;">
            <div style="font-size:13px;color:#64748b;">
                Total misclassifications
            </div>
            <div style="font-size:25px;font-weight:bold;">
                {total}
            </div>
        </div>

        <div style="flex:1;min-width:140px;background:#fee2e2;
                    padding:15px;border-radius:10px;">
            <div style="font-size:13px;color:#991b1b;">
                Bearish → Bullish
            </div>
            <div style="font-size:25px;font-weight:bold;">
                {total_bear}
            </div>
        </div>

        <div style="flex:1;min-width:140px;background:#dcfce7;
                    padding:15px;border-radius:10px;">
            <div style="font-size:13px;color:#166534;">
                Bullish → Bearish
            </div>
            <div style="font-size:25px;font-weight:bold;">
                {total_bull}
            </div>
        </div>

    </div>
    """


# =====================================================
# 4. POPULATE TWEET DROPDOWN
# =====================================================

def update_tweet_options():

    dataset = get_current_dataset()

    options = []

    for i, row in dataset.iterrows():

        tweet = str(row[TEXT_COL])

        short_text = (
            tweet[:75] + "..."
            if len(tweet) > 75 else tweet
        )

        actual = str(row[ACTUAL_COL]).capitalize()
        predicted = str(row[PREDICTED_COL]).capitalize()

        option_label = (
            f"#{i} | {actual} → {predicted} | {short_text}"
        )

        options.append((option_label, i))

    tweet_selector.options = options

    if options:
        tweet_selector.value = options[0][1]

    update_counter()


def update_counter():

    dataset = get_current_dataset()

    if len(dataset) == 0:
        tweet_counter.value = "<b>No tweets available.</b>"
        return

    position = tweet_selector.value

    tweet_counter.value = f"""
    <div style="margin:8px 0;color:#64748b;">
        Tweet <b>{position + 1}</b> of <b>{len(dataset)}</b>
    </div>
    """


# =====================================================
# 5. NAVIGATION
# =====================================================

def navigate_tweet(direction):

    dataset = get_current_dataset()

    if len(dataset) == 0:
        return

    current = tweet_selector.value

    new_position = max(
        0,
        min(len(dataset) - 1, current + direction)
    )

    tweet_selector.value = new_position

    update_counter()


previous_button.on_click(
    lambda button: navigate_tweet(-1)
)

next_button.on_click(
    lambda button: navigate_tweet(1)
)


# =====================================================
# 6. INTERACTIVE EXPLANATION VISUALIZATION
# =====================================================

def display_explanation(result, row):

    actual = str(row[ACTUAL_COL]).capitalize()
    predicted = str(row[PREDICTED_COL]).capitalize()

    tokens = result["tokens"]
    scores = result["attributions"]

    # Read probabilities from your existing DataFrame
    probabilities = {
        label: float(row[label])
        for label in SENTIMENT_COLS
    }

    max_score = max(
        (abs(score) for score in scores),
        default=0
    ) or 1

    # ---------------------------------
    # Classification status
    # ---------------------------------

    correct = actual.lower() == predicted.lower()

    status_color = "#dc2626" if not correct else "#16a34a"

    status_text = (
        "MISCLASSIFICATION"
        if not correct else "CORRECT"
    )

    # ---------------------------------
    # Probability bars
    # ---------------------------------

    probability_html = ""

    colors = {
        "negative": "#dc2626",
        "neutral": "#64748b",
        "positive": "#16a34a"
    }

    for label in SENTIMENT_COLS:

        score = probabilities[label]

        probability_html += f"""
        <div style="margin:13px 0;">

            <div style="display:flex;justify-content:space-between;
                        margin-bottom:5px;">

                <b>{label.capitalize()}</b>
                <b>{score * 100:.2f}%</b>

            </div>

            <div style="height:12px;background:#e5e7eb;
                        border-radius:10px;overflow:hidden;">

                <div style="height:12px;
                            width:{score * 100:.2f}%;
                            background:{colors[label]};
                            border-radius:10px;">
                </div>

            </div>
        </div>
        """

    # ---------------------------------
    # Token attribution
    # ---------------------------------

    token_html = ""

    for token, score in zip(tokens, scores):

        # Ignore special tokens in the displayed tweet
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        # Preserve WordPiece joining
        if token.startswith("##"):
            word = token[2:]
            margin = "0 2px 0 -3px"
        else:
            word = token
            margin = "0 2px"

        intensity = abs(score) / max_score

        alpha = 0.12 + 0.78 * intensity

        if score > 0:
            background = f"rgba(22,163,74,{alpha:.2f})"
        elif score < 0:
            background = f"rgba(220,38,38,{alpha:.2f})"
        else:
            background = "rgba(148,163,184,0.15)"

        token_html += f"""
        <span title="Attribution: {score:+.5f}"
              style="display:inline-block;
                     background:{background};
                     padding:6px 5px;
                     margin:{margin};
                     border-radius:5px;
                     cursor:help;
                     font-size:17px;">
            {escape(word)}
        </span>
        """

    # ---------------------------------
    # Most influential tokens
    # ---------------------------------

    ranked_tokens = sorted(
        [
            (token, score)
            for token, score in zip(tokens, scores)
            if token not in ["[CLS]", "[SEP]", "[PAD]"]
        ],
        key=lambda x: abs(x[1]),
        reverse=True
    )[:10]

    ranking_html = ""

    for token, score in ranked_tokens:

        color = "#16a34a" if score > 0 else "#dc2626"

        ranking_html += f"""
        <tr>
            <td style="padding:9px;border-bottom:1px solid #eee;">
                {escape(token)}
            </td>

            <td style="padding:9px;text-align:right;
                       color:{color};font-weight:bold;
                       border-bottom:1px solid #eee;">
                {score:+.5f}
            </td>
        </tr>
        """

    # ---------------------------------
    # Full dashboard
    # ---------------------------------

    dashboard_html = f"""
    <div style="font-family:Arial,sans-serif;
                max-width:950px;color:#1f2937;">

        <div style="background:#f8fafc;padding:20px;
                    border-radius:12px;margin:15px 0;">

            <div style="font-size:12px;color:#64748b;">
                CLASSIFICATION RESULT
            </div>

            <div style="font-size:27px;font-weight:bold;
                        margin:8px 0;">
                {escape(predicted.upper())}
            </div>

            <span style="background:{status_color};
                         color:white;padding:5px 10px;
                         border-radius:5px;font-size:12px;">
                {status_text}
            </span>

            <p style="margin-top:15px;">
                Actual sentiment: <b>{escape(actual)}</b>
                <br>
                Predicted sentiment: <b>{escape(predicted)}</b>
                <br>
                Prediction confidence:
                <b>{float(row["pred_top_score"]) * 100:.2f}%</b>
            </p>

        </div>

        <div style="border:1px solid #e5e7eb;
                    border-radius:12px;padding:20px;margin:15px 0;">

            <h3>1. Class probabilities</h3>

            {probability_html}

        </div>

        <div style="border:1px solid #e5e7eb;
                    border-radius:12px;padding:20px;margin:15px 0;">

            <h3>2. Word-level explanations</h3>

            <p style="font-size:18px;line-height:2.5;">
                {token_html}
            </p>

            <div style="display:flex;gap:15px;flex-wrap:wrap;
                        font-size:13px;">

                <span style="color:#16a34a;">
                    ■ Supports predicted class
                </span>

                <span style="color:#dc2626;">
                    ■ Opposes predicted class
                </span>

            </div>

            <p style="font-size:12px;color:#64748b;">
                Hover over a token to see its attribution score.
                Positive and negative refer to contribution toward
                or against the predicted class, not sentiment polarity.
            </p>

        </div>

        <div style="border:1px solid #e5e7eb;
                    border-radius:12px;padding:20px;margin:15px 0;">

            <h3>3. Most influential tokens</h3>

            <table style="width:100%;border-collapse:collapse;">

                <tr style="background:#f8fafc;">
                    <th style="padding:9px;text-align:left;">Token</th>
                    <th style="padding:9px;text-align:right;">
                        Attribution
                    </th>
                </tr>

                {ranking_html}

            </table>

        </div>

        <div style="color:#64748b;font-size:12px;margin:15px 0;">
            Integrated Gradients convergence delta:
            {result.get("convergence_delta", float("nan")):.6f}
        </div>

    </div>
    """

    display(HTML(dashboard_html))


# =====================================================
# 7. RUN INTEGRATED GRADIENTS ON SELECTED TWEET
# =====================================================

def analyze_selected_tweet(button=None):

    with analysis_output:

        clear_output(wait=True)

        dataset = get_current_dataset()

        if len(dataset) == 0:
            print("No misclassified tweets in this category.")
            return

        row = dataset.iloc[tweet_selector.value]

        tweet = str(row[TEXT_COL])

        print("Analyzing selected tweet...")

        try:

            result = explain_tweet(tweet)

            # Save convergence delta for the dashboard
            # Add this field to explain_tweet's return dictionary
            # if you have not already done so.
            result["convergence_delta"] = result.get(
                "convergence_delta", float("nan")
            )

            clear_output(wait=True)

            display_explanation(result, row)

        except Exception as e:

            clear_output(wait=True)

            print("Error:", str(e))


analyze_button.on_click(analyze_selected_tweet)


# =====================================================
# 8. UPDATE WHEN CATEGORY CHANGES
# =====================================================

def category_changed(change):

    if change["name"] == "value":

        update_tweet_options()

        with analysis_output:
            clear_output(wait=True)


category_selector.observe(
    category_changed,
    names="value"
)

tweet_selector.observe(
    lambda change: update_counter(),
    names="value"
)


# =====================================================
# 9. DISPLAY DASHBOARD
# =====================================================

update_dataset_stats()
update_tweet_options()

dashboard = widgets.VBox([

    widgets.HTML("""
        <h1 style="margin-bottom:5px;">
            FinBERT Explainability Dashboard
        </h1>

        <p style="color:#64748b;">
            Explore misclassified financial tweets and investigate
            which tokens influence FinBERT's predictions.
        </p>
    """),

    dataset_stats,

    widgets.HTML("<h3>Dataset selection</h3>"),

    category_selector,

    tweet_selector,

    widgets.HBox([
        previous_button,
        next_button
    ]),

    tweet_counter,

    analyze_button,

    analysis_output

])



In [15]:
display(dashboard)